# CALM-VAD — local run (VS Code notebook)

Runs on **your PC**, in **your `.venv`**, so you watch every step happen in
this notebook — no background jobs, no Colab. Pick the kernel
**`Python (sentrix .venv)`** top-right before running (or `.venv\Scripts\python.exe`).

Working directory should be `d:\Projects\Website\sentrix` (this file's folder's parent).

## 1 · Environment check

In [ ]:
import os, sys
os.chdir(r'd:\Projects\Website\sentrix')     # <- edit if your checkout lives elsewhere
print('cwd:', os.getcwd())
print('python:', sys.executable, sys.version.split()[0])
sys.path.insert(0, os.getcwd())
import calm
print('calm package OK, from', calm.__file__)

In [ ]:
# 22 checks on the M1..M4 maths, no data needed (~2s)
import subprocess
print(subprocess.run([sys.executable, '-m', 'calm.selftest'], text=True, capture_output=True).stdout)

## 2 · ShanghaiTech — already downloaded, replay the report

You already ran this once (took a few minutes). This cell just reads the
saved report so you can see the real numbers without waiting again. To
re-run it live and watch it compute, use the cell after.

In [ ]:
import json
r = json.load(open('results/calm_report_shanghaitech.json'))
print(f"ShanghaiTech: {r['n_calib']} calib / {r['n_test']} test clips\n")
for s in r['streams']:
    faph = {k.split('=')[-1]: v for k, v in s.items() if k.startswith('faph@')}
    print(f"  {s['label']:<26} AUC {s['frame_auc']:.3f}  eventF1 {s['event']['f1_avg']:.3f}  FAPH {faph}")
c = r['calibration']
print(f"\n  calibration ECE  : {c['ece_raw']:.4f} -> {c['ece_cal']:.4f}")
print(f"  decision cost    : {r['cost']['decision_layer_ms_per_frame_mean']:.3f} ms/frame")

**Want to watch it compute live instead of reading the cache?** Uncomment and run
(takes a few minutes — you'll see the `[inspect]` tree print immediately, then it
scores 79k frames twice, so it goes quiet for a while before the report appears):

In [ ]:
# for line in subprocess.Popen(
#         [sys.executable, '-m', 'calm.harness', '--auto', 'data/raw/shanghaitech',
#          '--tag', 'shanghaitech', '--fps', '24', '--save-generic', 'data/pose/shanghaitech.json'],
#         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1).stdout:
#     print(line, end='')

## 3 · UBnormal (STG-NF release) — already on disk, run it live now

Same zip as ShanghaiTech, a different subfolder. Ground-truth polarity is
**inverted** from ShanghaiTech (verified against the raw files: every
`normal_*` label file is uniformly 1.0, every `abnormal_*` file is a 0/1 mix
with a duration that varies clip to clip) — `calm.datasets.load_ubnormal_stgnf`
handles the inversion. 211 test clips, 91{,}318 frames, 199 gt events, 53
purely-normal clips. No download needed — this streams live right now.

In [ ]:
POSE = 'data/raw/shanghaitech/data/UBnormal/pose/test'
GT   = 'data/raw/shanghaitech/data/UBnormal/gt'
proc = subprocess.Popen(
    [sys.executable, '-m', 'calm.harness', '--ubnormal-stgnf', POSE, GT,
     '--tag', 'ubnormal', '--fps', '30', '--save-generic', 'data/pose/ubnormal.json'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n[exit code]', proc.returncode)

## 4 · CHAD (Charlotte Anomaly Dataset) — already on disk, run it live

CHAD ships pose natively (bbox + 17-keypoint COCO, per-frame pickles) — no
video, no GPU, ever. Downloaded automatically (single Drive file, no manual
step) to `data/raw/chad/`: `annotations/*.pkl` + `anomaly_labels/*.npy` +
`splits/*.txt`. 134 official test clips, 126{,}475 frames, 190 gt events.
Real detector NaNs (occluded joints) are sanitised by `load_chad`.

In [ ]:
proc = subprocess.Popen(
    [sys.executable, '-m', 'calm.harness', '--chad', 'data/raw/chad',
     '--tag', 'chad', '--fps', '30', '--save-generic', 'data/pose/chad.json'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n[exit code]', proc.returncode)

## 5 · HR-Avenue — get the data (one manual step, then everything is live below)

MoCoDAD publishes HR-Avenue poses + ground truth on Google Drive as a
**folder** with hundreds of files — `gdown` cannot fetch that automatically
(50-file limit). One-time manual step:

1. Open <https://drive.google.com/drive/folders/1aUDiyi2FCc6nKTNuhMvpGG_zLZzMMc83>
2. Right-click **Avenue** → **Download** (Google zips it for you, a couple hundred MB)
3. Save the zip as exactly:  `d:\Projects\Website\sentrix\data\raw\hr_avenue.zip`

Then run the cells below — from here on everything runs and prints live.

In [ ]:
import zipfile, tarfile
zpath = 'data/raw/hr_avenue.zip'
assert os.path.exists(zpath), f'put the downloaded zip at {zpath} first (see markdown above)'
dest = 'data/raw/hr_avenue'
os.makedirs(dest, exist_ok=True)
print('extracting', os.path.getsize(zpath)/1e6, 'MB ...')
(zipfile.ZipFile(zpath).extractall(dest) if zipfile.is_zipfile(zpath)
 else tarfile.open(zpath).extractall(dest))
print('done ->', dest)

## 6 · Inspect the tree (confirms the layout before we trust it)

In [ ]:
print(subprocess.run([sys.executable, '-m', 'calm.datasets', 'data/raw/hr_avenue'],
                     text=True, capture_output=True).stdout)

## 7 · Run the full harness — LIVE (streams output as it happens)

This is the real computation: load poses → M1 reliability → M2 evidence fusion
→ M3 calibration → M4 risk control → baselines → the 5-axis report. Every
line below appears the moment it's printed, not after the fact.

In [ ]:
proc = subprocess.Popen(
    [sys.executable, '-m', 'calm.harness', '--auto', 'data/raw/hr_avenue',
     '--tag', 'hr_avenue', '--fps', '25', '--save-generic', 'data/pose/hr_avenue.json'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n[exit code]', proc.returncode)

## 8 · Combined comparison — all datasets run so far, side by side

In [ ]:
import glob, json
for jp in sorted(glob.glob('results/calm_report_*.json')):
    r = json.load(open(jp))
    print(f"\n=== {r['tag']} ===")
    for s in r['streams']:
        if 'CALM-VAD (M1' in s['label'] or 'weighted-sum' in s['label']:
            print(f"  {s['label']:<26} AUC {s['frame_auc']:.3f}  eventF1 {s['event']['f1_avg']:.3f}")
    c = r['calibration']
    print(f"  ECE {c['ece_raw']:.4f} -> {c['ece_cal']:.4f}   "
          f"decision {r['cost']['decision_layer_ms_per_frame_mean']:.2f} ms/frame")

## 9 · Cross-dataset: fit on ShanghaiTech, test on HR-Avenue

Uses the `data/pose/*.json` files saved above — the paper's generalisation axis.

In [ ]:
A = json.load(open('data/pose/shanghaitech.json'))
B = json.load(open('data/pose/hr_avenue.json'))
for x in A['clips']: x['split'] = 'calib'
for x in B['clips']: x['split'] = 'test'
json.dump({'fps': A['fps'], 'clips': A['clips'] + B['clips']}, open('data/pose/sht_to_avenue.json', 'w'))
proc = subprocess.Popen([sys.executable, '-m', 'calm.harness', '--generic',
                         'data/pose/sht_to_avenue.json', '--tag', 'sht_to_avenue'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end='')